In [ ]:
%pip install numpy pandas scikit-learn xgboost

     ---------------------------------------- 0.0/20.7 MB ? eta -:--:--
     ------------------ --------------------- 9.7/20.7 MB 57.2 MB/s eta 0:00:01
     ------------------------------- ------- 16.5/20.7 MB 43.4 MB/s eta 0:00:01
     ---------------------------------------- 20.7/20.7 MB 35.1 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + c:\Users\Harsh\AppData\Local\Programs\Python\Python315\python.exe C:\Users\Harsh\AppData\Local\Temp\pip-install-v8vxapvw\numpy_1594314d2e544853a6ea4eafa08987de\vendored-meson\meson\meson.py setup C:\Users\Harsh\AppData\Local\Temp\pip-install-v8vxapvw\numpy_1594314d2e544853a6ea4eafa08987de C:\Users\Harsh\AppData\Local\Temp\pip-install-v8vxapvw\numpy_1594314d2e544853a6ea4eafa08987de\.mesonpy-51yh4vvy -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\Harsh\AppData\Local\Temp\pip-install-v8vxapvw\numpy_1594314d2e544853a6ea4eafa08987de\.mesonpy-51yh4vvy\meson-python-native-file.ini
      The Meson build system
      Version: 1.9.2
      Source dir: C:\Users\Harsh\AppData\Local\Temp\pip-install-v8vxapvw\numpy_1594314d2e544853a6ea4eafa08987de
      Build dir: C:\Users\Harsh\AppData\Local\Temp\pip-install-v8vx

In [ ]:
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

file_path = r'balanced_annotations_train_aggregated.csv'
base_features_path = r'balanced_features\balanced_features\video_without_context\train'

X_raw_list = []
y_raw_list = []
missing_files = []

try:
    df_annotations = pd.read_csv(file_path)
    print("successfully loaded file")
except FileNotFoundError:
    print(f"Error: Could not find the file at the specified path.")
except Exception as e:
    print(f"An error occurred: {e}")

for index, row in df_annotations.iterrows():
    video_title = str(row['file'])

    file_path = os.path.join(base_features_path, video_title.replace(".mp4", ""), f"{video_title.replace(".mp4", "")}.csv")
    #print(file_path)
    if os.path.exists(file_path):
        df_features = pd.read_csv(file_path)

        cols_to_drop = ['frame', 'timestamp', 'face_id', 'confidence', 'success']
        df_features = df_features.drop(columns=[c for c in cols_to_drop if c in df_features.columns])

        X_raw_list.append(df_features.values)

        y_raw_list.append(row.to_dict())
    else:
        missing_files.append(video_title)

print(f"Successfully loaded {len(X_raw_list)} valid feature files.")
if missing_files:
    print(f"Warning: Could not find {len(missing_files)} files. First few missing: {missing_files[:3]}")

# median length of X
median_length = int(np.median([x.shape[0] for x in X_raw_list]))
TARGET_TIME_STEPS = median_length
print(f"\nStandardizing all sequences to {TARGET_TIME_STEPS} time steps...")

X_raw_uniform = []

# make all the input of same shape
for x in X_raw_list:
    current_steps = x.shape[0]
    num_features = x.shape[1]

    if current_steps > TARGET_TIME_STEPS:
        x_uniform = x[:TARGET_TIME_STEPS, :]
    elif current_steps < TARGET_TIME_STEPS:
        padding = np.zeros((TARGET_TIME_STEPS - current_steps, num_features))
        x_uniform = np.vstack((x, padding))
    else:
        x_uniform = x

    X_raw_uniform.append(x_uniform)

X_raw = np.array(X_raw_uniform)
Y_raw_df = pd.DataFrame(y_raw_list)

print("\ndata status")
print(f"X_raw shape: {X_raw.shape} ")
print(f"Y_raw_df shape: {Y_raw_df.shape}")

successfully loaded file
Successfully loaded 1178 valid feature files.

Standardizing all sequences to 252 time steps...

=== FINAL DATA STATUS ===
X_raw shape: (1178, 252, 333) -> (Samples, Time Steps, Features)
Y_raw_df shape: (1178, 2) -> (Samples, Columns)

Y_raw_df Columns available:
['file', 'engaged']


In [ ]:
#transformation strategies
def apply_temporal_aggregation(X):
    mean_f = np.mean(X, axis=1)
    var_f = np.var(X, axis=1)
    max_f = np.max(X, axis=1)
    return np.concatenate([mean_f, var_f, max_f], axis=1)

def apply_dynamic_features(X):
    diff_f = np.diff(X, axis=1)
    return np.mean(np.abs(diff_f), axis=1)

def apply_sliding_window(X, y, window_size=25, step_size=20):
    X_win_list, y_win_list = [], []
    num_samples = X.shape[0]
    seq_length = X.shape[1]

    is_dataframe = isinstance(y, pd.DataFrame)

    for i in range(num_samples):
        for w_start in range(0, seq_length - window_size + 1, step_size):
            window = X[i, w_start : w_start + window_size]
            X_win_list.append(window.flatten())

            if is_dataframe:
                y_win_list.append(y.iloc[i])
            else:
                y_win_list.append(y[i])

    if is_dataframe:
        return np.array(X_win_list), pd.DataFrame(y_win_list).reset_index(drop=True)
    else:
        return np.array(X_win_list), np.array(y_win_list)


datasets = {}

datasets["Temporal Aggregation"] = {
    "X": apply_temporal_aggregation(X_raw),
    "Y": Y_raw_df.copy(),
}

datasets["Dynamic Features"] = {
    "X": apply_dynamic_features(X_raw),
    "Y": Y_raw_df.copy(),
}

X_tr_win, Y_tr_win = apply_sliding_window(X_raw, Y_raw_df)

datasets["Sliding Window"] = {
    "X": X_tr_win,
    "Y": Y_tr_win,
}


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

#keep 90% of variance
pca_variance_ratio = 0.90

reduced_datasets = {}

for name, data in datasets.items():
    X = data["X"]
    Y = data["Y"]

    print(f"Processing {name}")
    print(f"  Original shape: {X.shape}")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=pca_variance_ratio)
    X_pca = pca.fit_transform(X_scaled)

    reduced_datasets[f"{name} (PCA)"] = {
        "X": X_pca,
        "Y": Y,
        "pca_object": pca 
    }

    print(f"  Reduced shape:  {X_pca.shape}")
    print(f"  Components retained: {pca.n_components_}\n")


print("Saving reduced datasets")

for name, data in reduced_datasets.items():
    X_pca = data["X"]
    Y = data["Y"]
    
    num_components = X_pca.shape[1]
    feature_columns = [f"PC{i+1}" for i in range(num_components)]
    
    df = pd.DataFrame(X_pca, columns=feature_columns)

    Y_df = pd.DataFrame(Y).reset_index(drop=True)

    if len(Y_df.columns) == 1 and Y_df.columns[0] == 0:
        Y_df.columns = ["Target"]
        
    df = pd.concat([df, Y_df], axis=1)
    
    clean_name = name.replace(" ", "_").replace("(", "").replace(")", "").lower()
    filename = f"{clean_name}.csv"
    
    df.to_csv(filename, index=False)
    print(f"Saved {filename}successfully.")

Processing Temporal Aggregation
  Original shape: (1178, 999)
  Reduced shape:  (1178, 12)
  Components retained: 12

Processing Dynamic Features
  Original shape: (1178, 333)
  Reduced shape:  (1178, 8)
  Components retained: 8

Processing Sliding Window
  Original shape: (11780, 3330)
  Reduced shape:  (11780, 11)
  Components retained: 11

Saving reduced datasets
Saved temporal_aggregation_pca.csvsuccessfully.
Saved dynamic_features_pca.csvsuccessfully.
Saved sliding_window_pca.csvsuccessfully.


In [17]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

folder_path = 'task_2_dataset_after_pca' 

file_names = [
    "temporal_aggregation_pca.csv",
    "dynamic_features_pca.csv",
    "sliding_window_pca.csv"
]

param_grids = {
    "Logistic Regression": {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['liblinear', 'lbfgs']
    },
    "SVM": {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto', 0.1],
        'kernel': ['rbf']
    },
    "Random Forest": {
        'n_estimators': [50, 100],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    },
    "XGBoost": {
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 6],
        'n_estimators': [100, 200]
    },
    "MLP": {
        'hidden_layer_sizes': [(64, 32), (128, 64)],
        'learning_rate_init': [0.001, 0.01]
    }
}

def get_base_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "SVM": SVC(probability=True, random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42)
    }

final_results = []

for file_name in file_names:
    full_path = os.path.join(folder_path, file_name)
    dataset_name = file_name.replace(".csv", "").replace("_", " ")

    print(f"\nEvaluating: {dataset_name}")

    try:
        df = pd.read_csv(full_path)
    except FileNotFoundError:
        print(f"Error: Could not find {full_path}. Please check your folder_path and file_names.")
        continue

    X = df.drop(columns=['file', 'engaged'])
    y = df['engaged']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    base_models = get_base_models()

    for model_name, model in base_models.items():
        grid_search = GridSearchCV(model, param_grids[model_name], cv=3, scoring='accuracy', n_jobs=-1)
        grid_search.fit(X_train, y_train)
      
        best_model = grid_search.best_estimator_

        train_preds = best_model.predict(X_train)
        train_acc = accuracy_score(y_train, train_preds)
        train_f1 = f1_score(y_train, train_preds, average='weighted')

        test_preds = best_model.predict(X_test)
        test_acc = accuracy_score(y_test, test_preds)
        test_f1 = f1_score(y_test, test_preds, average='weighted')

        # 4. Print formatted results for this run
        print(f"\n{model_name.upper()}")
        print(f"  Best Params: {grid_search.best_params_}")
        print(f"  TRAIN  :Accuracy: {train_acc:.4f} , F1-Score: {train_f1:.4f}")
        print(f"  TEST  : Accuracy: {test_acc:.4f} , F1-Score: {test_f1:.4f}")

        # 5. Store metrics for the final summary table
        final_results.append({
            'Temporal Strategy': dataset_name,
            'Model': model_name,
            'Test Accuracy': test_acc,
            'Test F1-Score': test_f1
        })

print("\nComplete.")

if final_results:
    print("\nFINAL SUMMARY: ALL MODELS ACROSS ALL TEMPORAL STRATEGIES")

    results_df = pd.DataFrame(final_results)

    acc_table = results_df.pivot(index='Model', columns='Temporal Strategy', values='Test Accuracy')
    f1_table = results_df.pivot(index='Model', columns='Temporal Strategy', values='Test F1-Score')

    print("\nTEST ACCURACY")
    print(acc_table.round(4).to_string())

    print("\nTEST F1-SCORE")
    print(f1_table.round(4).to_string())


Evaluating: temporal aggregation pca

LOGISTIC REGRESSION
  Best Params: {'C': 0.1, 'solver': 'lbfgs'}
  TRAIN  :Accuracy: 0.5276 , F1-Score: 0.5246
  TEST  : Accuracy: 0.4958 , F1-Score: 0.4957

SVM
  Best Params: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
  TRAIN  :Accuracy: 0.9851 , F1-Score: 0.9851
  TEST  : Accuracy: 0.4873 , F1-Score: 0.4794

RANDOM FOREST
  Best Params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
  TRAIN  :Accuracy: 0.9735 , F1-Score: 0.9734
  TEST  : Accuracy: 0.4873 , F1-Score: 0.4868

XGBOOST
  Best Params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
  TRAIN  :Accuracy: 0.6518 , F1-Score: 0.6516
  TEST  : Accuracy: 0.5042 , F1-Score: 0.5027

MLP
  Best Params: {'hidden_layer_sizes': (128, 64), 'learning_rate_init': 0.001}
  TRAIN  :Accuracy: 0.9745 , F1-Score: 0.9745
  TEST  : Accuracy: 0.5000 , F1-Score: 0.4987

Evaluating: dynamic features pca

LOGISTIC REGRESSION
  Best Params: {'C': 0.01, 'solver': 'liblinear'}
  TRAI